# 6 — RL + JEPA (JEPA world model + banded actor-critic ensemble)

Like notebook 4, but the state representation is a **JEPA latent world model** instead of raw
attention-over-tokens: JEPA encoders learn `s_t -> z_t` from unlabelled trajectories (frames,
sensor, text), a predictor learns latent dynamics `z_t, a_t -> z_{t+1}`, and the actor-critic
members plan/act in latent space. Banded bagging gates which agents join the behaviour ensemble
by normalised return, with the same grace period. This is the DreamerV3 / TD-MPC2 family:
self-supervised world model + actor-critic, plus your band protocol for bias control.

In [1]:
import importlib.util, os, pathlib, sys
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None: break
    p = p.parent
if ai is None: raise RuntimeError('ai_service not found')
sys.path.insert(0, str(ai / 'training')); sys.path.insert(0, str(ai))
os.chdir(ai / 'notebooks')
# ── load the repo .env (BUDDY_SCALE, KAGGLE_API_TOKEN, …) BEFORE bootstrap ──
# bootstrap reads BUDDY_SCALE at import time, so this must run first. Uses
# python-dotenv when available, else a tiny built-in parser (Kaggle-safe).
def _find_dotenv(start):
    p = pathlib.Path(start).resolve()
    while p != p.parent:
        f = p / '.env'
        if f.is_file():
            return f
        p = p.parent
    return None

_env_file = _find_dotenv(ai)
try:
    from dotenv import load_dotenv
    load_dotenv(_env_file)
except ImportError:
    if _env_file:
        for _line in _env_file.read_text().splitlines():
            _line = _line.strip()
            if not _line or _line.startswith('#') or '=' not in _line:
                continue
            _k, _, _v = _line.partition('=')
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
print('[env]', _env_file or 'no .env found', '| BUDDY_SCALE =', os.environ.get('BUDDY_SCALE'),
      '| KAGGLE_API_TOKEN =', 'set' if os.environ.get('KAGGLE_API_TOKEN') else 'missing')
_missing = [m for m in ['torch'] if importlib.util.find_spec(m) is None]
if _missing:
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(_missing))
# bootstrap.py reads BUDDY_SCALE at import time; if an earlier run in this
# same kernel cached the module (e.g. before the .env was loaded), drop the
# stale copy so the current environment is honoured.
for _stale in ('training.bootstrap', 'bootstrap'):
    _m = sys.modules.get(_stale)
    if _m is not None and getattr(_m, 'BUDDY_SCALE', None) != os.environ.get('BUDDY_SCALE'):
        sys.modules.pop(_stale, None)
        print(f'[bootstrap] re-importing {_stale} (stale scale cache cleared)')
try:
    from training.bootstrap import *
    CFG = init(scale=os.environ.get('BUDDY_SCALE') or None)
except Exception as e:
    print('[bootstrap] unavailable:', e); CFG = {}
except Exception as e:
    print('[bootstrap] unavailable:', e); CFG = {}
SCALE = CFG.get('scale', os.environ.get('BUDDY_SCALE', 'demo'))
print('scale:', SCALE)

[env] /home/peter/Desktop/Buddy-Up/backend/.env | BUDDY_SCALE = smoke | KAGGLE_API_TOKEN = set


2026-09-16 16:49:14.503165: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-16 16:49:14.727931: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-09-16 16:49:18.836218: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


scale: smoke


In [2]:
import torch, torch.nn as nn, numpy as np
torch.manual_seed(4); np.random.seed(4)
BANDS = [0.37, 0.47, 0.57, 0.67, 0.71, 0.73, 0.77, 0.79, 0.81, 0.85, 0.90]
GRACE_EP = {'smoke': 2, 'demo': 5, 'full': 20}[SCALE]
N = {'smoke': 3, 'demo': 5, 'full': 10}[SCALE]
from batch_data import has_batch, batch_meta, load_tensors
_MB = batch_meta() if has_batch() else None
print('REAL transitions:', _MB['source'] if _MB else 'synthetic toy env')
OBS, Z, NA = 48, 32, (_MB['na'] if _MB else 4)

class JEPAWorldModel(nn.Module):
    """Encoder obs->z (EMA target), latent predictor (z,a)->z'. Loss = MSE in latent space."""
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(OBS, 128), nn.GELU(), nn.Linear(128, Z))
        self.tgt = nn.Sequential(nn.Linear(OBS, 128), nn.GELU(), nn.Linear(128, Z))
        self.dyn = nn.Sequential(nn.Linear(Z + NA, 128), nn.GELU(), nn.Linear(128, Z))
        self._ema(0.0)
    @torch.no_grad()
    def _ema(self, m=0.996):
        for a, b in zip(self.tgt.parameters(), self.enc.parameters()): a.data.mul_(m).add_(b.data, alpha=1-m)
    def loss(self, o, o_next, a_oh):
        with torch.no_grad(): zt = self.tgt(o_next)
        zp = self.dyn(torch.cat([self.enc(o), a_oh], -1))
        return ((zp - zt) ** 2).mean()

class LatentActorCritic(nn.Module):
    def __init__(self, wm):
        super().__init__(); self.wm = wm
        self.pi = nn.Linear(Z, NA); self.v = nn.Linear(Z, 1)
    def act(self, o):
        with torch.no_grad():
            z = self.wm.enc(o)
            return int(torch.softmax(self.pi(z), -1).multinomial(1))

REAL transitions: CartPole-v1 random rollouts, obs tiled 4->48


In [3]:
# Toy continuous-obs env with latent structure. Real: DM-Control / MetaWorld / BuddyUp workout logs.
class LatentToy:
    def __init__(self): self.s = np.random.randn(8)
    def reset(self):
        self.s = np.random.randn(8); return torch.tensor(np.tile(self.s, 6)[:OBS], dtype=torch.float32).unsqueeze(0)
    def step(self, a):
        target = int(self.s[:4].argmax()) % NA
        r = float(a == target)
        self.s = np.random.randn(8)
        return torch.tensor(np.tile(self.s, 6)[:OBS], dtype=torch.float32).unsqueeze(0), r, True, {}
wm = JEPAWorldModel()
agents = [LatentActorCritic(wm) for _ in range(N)]  # shared world model, diverse actors (standard)
opt_wm = torch.optim.Adam(list(wm.enc.parameters()) + list(wm.dyn.parameters()), lr=1e-3)
opt_ac = [torch.optim.Adam(list(a.pi.parameters()) + list(a.v.parameters()), lr=2e-3) for a in agents]
EPISODES = {'smoke': 150, 'demo': 250, 'full': 1500}[SCALE]
WINDOW = {'smoke': 50, 'demo': 100, 'full': 100}[SCALE]  # rolling window for band gating
scores = {i: [] for i in range(N)}; joined = {}
_TB = load_tensors('o', 'a', 'r', 'o2') if _MB is not None else None
import torch.nn.functional as F
env = LatentToy()
for ep in range(EPISODES):
    for i, ag in enumerate(agents):
        o = env.reset()
        a = ag.act(o); o2, r, _, _ = env.step(a)
        if _TB is not None:  # REAL dynamics: CartPole transition batch
            bi = torch.randint(0, len(_TB['o']), (32,))
            aoh = torch.zeros(32, NA); aoh[torch.arange(32), _TB['a'][bi]] = 1.0
            opt_wm.zero_grad(); wl = wm.loss(_TB['o'][bi], _TB['o2'][bi], aoh)
            wl.backward(); opt_wm.step(); wm._ema()
        else:
            aoh = torch.zeros(1, NA); aoh[0, a] = 1.0
            opt_wm.zero_grad(); wl = wm.loss(o, o2, aoh); wl.backward(); opt_wm.step(); wm._ema()
        with torch.no_grad(): z = wm.enc(o)
        logits, v = ag.pi(z), ag.v(z)
        dist = torch.distributions.Categorical(logits=logits)
        act = dist.sample(); adv = torch.tensor(float(r)) - v.detach()
        loss = -(dist.log_prob(act) * adv) + F.mse_loss(v, torch.tensor([[float(r)]]))
        opt_ac[i].zero_grad(); loss.backward(); opt_ac[i].step()
        scores[i].append(r)
    if (ep + 1) % 25 == 0:
        for i in range(N):
            acc = float(np.mean(scores[i][-WINDOW:]))
            b = max([t for t in BANDS if acc >= t], default=None)
            if b and i not in joined:
                joined[i] = (b, ep); print(f'ep{ep}: agent {i} return={acc:.2f} joins {int(b*100)}% (wm-loss={wl.item():.4f})')
print('joined:', joined)

ep24: agent 0 return=0.44 joins 37% (wm-loss=0.0002)
ep24: agent 1 return=0.48 joins 47% (wm-loss=0.0002)
ep24: agent 2 return=0.44 joins 37% (wm-loss=0.0002)


joined: {0: (0.37, 24), 1: (0.47, 24), 2: (0.37, 24)}


In [4]:
# Bagged latent policy over grace-eligible agents
elig = [i for i, (_, e0) in joined.items() if EPISODES - e0 >= GRACE_EP] or list(range(N))
wins = []
for _ in range(200):
    o = env.reset()
    with torch.no_grad():
        votes = [int(agents[i].pi(wm.enc(o)).argmax(1)) for i in elig]
    a = max(set(votes), key=votes.count); _, r, _, _ = env.step(a); wins.append(r)
print(f'eligible={elig} bagged return={np.mean(wins):.3f}')
class LatentPolicy(nn.Module):
    def __init__(self, wm, ag): super().__init__(); self.e = wm.enc; self.p = ag.pi
    def forward(self, o): return self.p(self.e(o))
lp = LatentPolicy(wm, agents[elig[0]]).eval()
torch.onnx.export(lp, torch.randn(1, OBS), '../models/rl_jepa_policy.onnx',
    input_names=['obs'], output_names=['action_logits'], dynamic_axes={'obs': {0: 'batch'}})
print('exported ../models/rl_jepa_policy.onnx')

eligible=[0, 1, 2] bagged return=0.480


/tmp/ipykernel_2893845/1702404498.py:14: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(lp, torch.randn(1, OBS), '../models/rl_jepa_policy.onnx',


[torch.onnx] Obtain model graph for `LatentPolicy([...]` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `LatentPolicy([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
exported ../models/rl_jepa_policy.onnx


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


## Data, scrapers vs platforms (notebook 6)

| Need | Recommendation |
|---|---|
| World-model data | Unlabelled trajectories FIRST: BuddyUp workout/form video + sensor + engagement sequences; public: Ego4D, Something-Something v2, DM-Control / MetaWorld logs, Atari (ALE). JEPA shines exactly where labels are scarce. |
| Reward data | Same trajectories + sparse outcomes (completion, click, form-score). No scraping — rewards must come from real user outcomes. |
| Scrapers/bots | Not applicable. A bot cannot generate physics/contact dynamics; use simulators (Mujoco, Isaac Gym, Habitat) for synthetic rollouts instead. |
| Platforms | DreamerV3 / TD-MPC2 / JEPA official repos (algorithm), CleanRL + SB3 (baselines), Ray RLlib / EnvPool (throughput), W&B (return + wm-loss curves). Managed: SageMaker RL, Vertex AI, Anyscale. No vendor sells 'RL+JEPA+banded-bagging' — this notebook is the recipe; vendors supply the GPUs. |

**Suggested reading:** I-JEPA (Assran et al. 2023), V-JEPA (Meta 2024), DreamerV3 (Hafner et al. 2023), TD-MPC2 (Hansen et al. 2023).